# Baseline SLM Benchmark - Mission 02.2 (Kaggle-only)

Evaluates three non-fine-tuned Instruct models in FP16 on six Software
Engineering tasks. Official baseline runs are executed on Kaggle only.

**Scope and safety**

- No Google Drive, no Drive mounting, no Colab and no cloud synchronisation.
- Benchmark data is read **only** from `/kaggle/input/...` (read-only).
- Temporary files, checkpoints, results and the export ZIP are written under
  `/kaggle/working/`.
- One model is loaded at a time, on exactly one GPU (T4 16 GB target).
- `MODE="full"` is the official mode; `MODE="smoke_test"` is a separate
  reduced run (first `SMOKE_PER_TASK` rows per task).
- Generated code is **never executed** by this notebook: no isolated sandbox
  is available in the Kaggle runtime. Code-generation, refactoring and
  unit-test evaluation are recorded as `skipped` until a real isolated
  backend is provided.
- The temporary Kaggle filesystem does not survive disconnection. Export
  and download the ZIP from the Output pane after every model run.


## 0. Kaggle prerequisites (manual, before running)

1. Create a **private** Kaggle Dataset containing the benchmark folder, so
   that the mounted root has `benchmark/datasets/test/`,
   `benchmark/datasets/private/test/` and `benchmark/prompts/prompt_registry.py`.
   The prepared local bundle is `kaggle_upload/slm-benchmark-baseline/`.
2. Create a Kaggle Notebook from `benchmark/notebooks/baseline_benchmark.ipynb`.
3. Add inputs:
   - **Datasets:** your private benchmark dataset.
   - **Models (optional, recommended for Qwen):** `qwen-lm/qwen2.5-coder`
     with `1.5b-instruct` and `3b-instruct`. If absent, the notebook falls
     back to the Hugging Face model ID.
   - **Phi-4-mini:** the notebook uses `microsoft/Phi-4-mini-instruct` from
     Hugging Face by default. A community Kaggle mirror exists but is not
     official; only use it after verifying its files.
4. Notebook settings: **Accelerator = GPU T4 x1**, **Internet = On** (needed
   for the Hugging Face fallback and dependencies), **Persistence = Files only**.
5. Run the environment cell. If it reports `rouge_score` as `MISSING`, run
   `!pip install -q rouge_score` in a separate Kaggle cell and re-run the
   environment cell. Do not let code_explanation silently score `None`.
6. Run the cells in order. Start with `MODE="smoke_test"` and one model;
   then run `MODE="full"` for each model, one at a time. Download the ZIP
   from the Output pane after each model and before disconnecting.

The notebook never launches a run by itself: every code cell is executed
manually by you.


## 1. Environment verification (read-only)

This cell only prints versions and GPU information. It does not load a model,
run the benchmark or consume GPU compute.


In [ ]:
# Environment verification
import os, sys, platform

# Kaggle-only temporary storage: keep Hugging Face downloads under
# /kaggle/working as well, so benchmark artifacts and caches share the same
# disposable filesystem. This must be set before transformers is imported.
os.environ.setdefault("HF_HOME", "/kaggle/working/.hf_cache")
os.environ.setdefault("HF_HUB_CACHE", "/kaggle/working/.hf_cache/hub")

print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device count:", torch.cuda.device_count())
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.2f} GB")
except Exception as e:
    print("PyTorch check failed:", e)

for pkg in ["transformers", "accelerate", "datasets", "evaluate", "rouge_score", "pandas", "numpy"]:
    try:
        m = __import__(pkg)
        print(pkg, getattr(m, "__version__", "ok"))
    except Exception as e:
        print(pkg, "MISSING:", e)


## 2. Kaggle storage paths (read-only input, writable working)

Benchmark data is read from `/kaggle/input/...`; nothing is written there.
Temporary files, checkpoints, results and the ZIP live under
`/kaggle/working/`. **Never assume `/kaggle/working/` survives after the
session ends** - download the ZIP before disconnecting.

Consistent result layout (Kaggle only):

```text
results/
├── smoke_test/
│   ├── qwen15/<RUN_ID>/*.jsonl, *_all_results.csv, metadata.json
│   ├── qwen3/<RUN_ID>/...
│   └── phi4mini/<RUN_ID>/...
├── baseline/                 # written by MODE="full"
│   ├── qwen15/<RUN_ID>/...
│   ├── qwen3/<RUN_ID>/...
│   └── phi4mini/<RUN_ID>/...
└── metadata/
    ├── run_<RUN_ID>.json
    ├── progress_<RUN_ID>.json
    ├── dataset_manifest.json
    ├── export_manifest.json
    └── SHA256SUMS.txt
```

- Input root: `/kaggle/input/<dataset-slug>/benchmark/...` (read-only).
- Working root: `/kaggle/working/` (temporary, writable).
- Results/checkpoints: `/kaggle/working/results/`.
- Local Windows (organise downloads only):
  `%USERPROFILE%\Downloads\slm-benchmark\`.


In [ ]:
# Kaggle-only storage paths. No Google Drive, no Colab, no cloud sync.
import glob
import os
import socket
import uuid
from datetime import datetime, timezone

IN_KAGGLE = os.path.isdir("/kaggle/working") or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if not IN_KAGGLE:
    raise RuntimeError(
        "This notebook is Kaggle-only. Open it in a Kaggle Notebook with a GPU "
        "accelerator; do not run it on Colab or on a local machine."
    )

PLATFORM = "kaggle"
INPUT_ROOT = "/kaggle/input"
WORKDIR = "/kaggle/working"
RESULTS_ROOT = os.path.join(WORKDIR, "results")
CHECKPOINTS_ROOT = RESULTS_ROOT
ZIP_DIR = WORKDIR

# Keep Hugging Face downloads under the disposable working tree as well.
# The export ZIP only contains results/, so the cache is never exported.
HF_CACHE_ROOT = os.path.join(WORKDIR, ".hf_cache")
os.environ.setdefault("HF_HOME", HF_CACHE_ROOT)
os.environ.setdefault("HF_HUB_CACHE", os.path.join(HF_CACHE_ROOT, "hub"))

TASKS = [
    "code_generation",
    "code_explanation",
    "bug_detection",
    "code_refactoring",
    "unit_test_generation",
    "code_completion",
]

def _valid_benchmark_root(root):
    # The dataset root must contain the model-facing test files, the private
    # references and the prompt registry. Nothing is written to this tree.
    test_dir = os.path.join(root, "benchmark", "datasets", "test")
    private_dir = os.path.join(root, "benchmark", "datasets", "private", "test")
    prompts = os.path.join(root, "benchmark", "prompts", "prompt_registry.py")
    if not (os.path.isdir(test_dir) and os.path.isdir(private_dir) and os.path.isfile(prompts)):
        return False
    return all(os.path.isfile(os.path.join(test_dir, f"{task}.jsonl")) for task in TASKS)

# A Kaggle dataset is mounted at /kaggle/input/<dataset-slug>/. The benchmark
# folder may be the dataset root or one or two levels below it. Only directory
# names and file presence are checked here.
_candidates = []
for _pattern in (
    os.path.join(INPUT_ROOT, "*", "benchmark"),
    os.path.join(INPUT_ROOT, "*", "*", "benchmark"),
    os.path.join(INPUT_ROOT, "*", "*", "*", "benchmark"),
):
    for _benchmark_dir in sorted(glob.glob(_pattern)):
        _root = os.path.dirname(_benchmark_dir)
        if _root not in _candidates and _valid_benchmark_root(_root):
            _candidates.append(_root)

# Explicit writable fallback for a manual upload, clearly flagged. Kaggle
# input datasets are read-only and are always preferred.
_working_fallback = os.path.join(WORKDIR, "slm-benchmark")
if not _candidates and _valid_benchmark_root(_working_fallback):
    _candidates.append(_working_fallback)

if not _candidates:
    raise FileNotFoundError(
        "Benchmark dataset not found under /kaggle/input. Attach a private "
        "Kaggle Dataset whose root contains benchmark/datasets/test/*.jsonl, "
        "benchmark/datasets/private/test/*.jsonl and "
        "benchmark/prompts/prompt_registry.py."
    )

BENCHMARK_ROOT = _candidates[0]
INPUT_IS_READ_ONLY = BENCHMARK_ROOT.startswith(INPUT_ROOT + os.sep)
if not INPUT_IS_READ_ONLY:
    print("WARNING: benchmark files are under /kaggle/working (writable).")
    print("         Prefer attaching them as a read-only /kaggle/input dataset.")

DATASETS_TEST = os.path.join(BENCHMARK_ROOT, "benchmark", "datasets", "test")
DATASETS_PRIVATE_TEST = os.path.join(BENCHMARK_ROOT, "benchmark", "datasets", "private", "test")
PROMPTS_DIR = os.path.join(BENCHMARK_ROOT, "benchmark", "prompts")

MODEL_DIR_NAMES = ["qwen15", "qwen3", "phi4mini"]
for _mode_dir in ("smoke_test", "baseline"):
    for _model_dir in MODEL_DIR_NAMES:
        os.makedirs(os.path.join(RESULTS_ROOT, _mode_dir, _model_dir), exist_ok=True)
os.makedirs(os.path.join(RESULTS_ROOT, "metadata"), exist_ok=True)

assert RESULTS_ROOT.startswith(WORKDIR + os.sep), RESULTS_ROOT
assert not RESULTS_ROOT.startswith(INPUT_ROOT + os.sep), RESULTS_ROOT

print("PLATFORM =", PLATFORM)
print("Benchmark root candidates =", _candidates)
print("BENCHMARK_ROOT =", BENCHMARK_ROOT)
print("INPUT_IS_READ_ONLY =", INPUT_IS_READ_ONLY)
print("RESULTS_ROOT =", RESULTS_ROOT)
print("CHECKPOINTS_ROOT =", CHECKPOINTS_ROOT)
print("ZIP_DIR =", ZIP_DIR)
print("HF_CACHE_ROOT =", HF_CACHE_ROOT)
print("LOCAL_DOWNLOAD_HINT =", os.path.join(os.environ.get("USERPROFILE", "%USERPROFILE%"), "Downloads", "slm-benchmark"))


## 3. Run configuration (MODE, RUN_ID, single-GPU, safety gate)

- `MODE="full"` - official run, all 670 test examples. It writes to
  `results/baseline/` to preserve the Mission 02.1 layout. `MODE="baseline"`
  is accepted as a legacy alias and normalised to `"full"`.
- `MODE="smoke_test"` - first `SMOKE_PER_TASK` rows per task, written to
  `results/smoke_test/`.
- Each `(MODE, model)` gets a unique `RUN_ID`; re-running the main loop in
  the same kernel resumes the same directory, a new session gets a new
  `RUN_ID`.
- `DEVICE_MAP={"": 0}` forces a single GPU even if Kaggle exposes two T4s.
- `EXECUTION_BACKEND="none"`: generated code is never executed. Tasks that
  require execution are generated but their evaluation is marked `skipped`.


In [ ]:
# Benchmark configuration (Kaggle-only)
import os
import random

MODEL_CHOICES = [
    "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    "Qwen/Qwen2.5-Coder-3B-Instruct",
    "microsoft/Phi-4-mini-instruct",
]

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

# Official mode is "full"; "baseline" is a legacy alias normalised below.
MODE = "full"
if MODE == "baseline":
    print("MODE='baseline' is a legacy alias; using MODE='full'.")
    MODE = "full"
SMOKE_PER_TASK = 3
assert MODE in ("smoke_test", "full"), MODE
MODE_DIR = {"full": "baseline", "smoke_test": "smoke_test"}
RESULTS_MODE_DIR = MODE_DIR[MODE]

GEN_CONFIG = {
    "temperature": 0.0,
    "do_sample": False,
    "top_p": 1.0,
    "max_new_tokens": 512,
    "repetition_penalty": 1.0,
}
SEED = 42
random.seed(SEED)

MODEL_META = {
    "Qwen/Qwen2.5-Coder-1.5B-Instruct": {"short": "qwen2.5-coder-1.5b", "model_dir": "qwen15", "trust_remote_code": False, "attn": None},
    "Qwen/Qwen2.5-Coder-3B-Instruct": {"short": "qwen2.5-coder-3b", "model_dir": "qwen3", "trust_remote_code": False, "attn": None},
    "microsoft/Phi-4-mini-instruct": {"short": "phi-4-mini", "model_dir": "phi4mini", "trust_remote_code": True, "attn": "eager"},
}

# Optional read-only Kaggle Models inputs. If a path is not attached, the
# notebook falls back to the Hugging Face model ID (Internet required).
MODEL_SOURCES = {
    "Qwen/Qwen2.5-Coder-1.5B-Instruct": [
        "/kaggle/input/qwen2.5-coder/transformers/1.5b-instruct/1",
    ],
    "Qwen/Qwen2.5-Coder-3B-Instruct": [
        "/kaggle/input/qwen2.5-coder/transformers/3b-instruct/1",
    ],
    "microsoft/Phi-4-mini-instruct": [
        # Community Kaggle mirror (not official). Used only if attached.
        "/kaggle/input/phi-4-mini-instruct/transformers/official-from-huggingface/1",
        "/kaggle/input/phi-4-mini-instruct/transformers/official-from-huggingface/1/phi-4-mini-instruct",
    ],
}

def resolve_model_source(model_id):
    for candidate in MODEL_SOURCES.get(model_id, []):
        if os.path.isfile(os.path.join(candidate, "config.json")):
            return candidate
    return model_id

MODEL_SOURCE = resolve_model_source(MODEL_ID)
assert MODEL_ID in MODEL_META
SHORT = MODEL_META[MODEL_ID]["short"]
MODEL_DIR = MODEL_META[MODEL_ID]["model_dir"]
assert MODEL_DIR in MODEL_DIR_NAMES, MODEL_DIR

# Single GPU only. Kaggle can expose two T4s; device_map {"": 0} prevents
# accelerate from sharding the model across both.
SINGLE_GPU = True
DEVICE_MAP = {"": 0} if SINGLE_GPU else "auto"

# Memory guard. No benchmark prompt reaches this bound; if one does, the row
# records input_truncated=True and the benchmark must be reviewed.
MAX_INPUT_TOKENS = 8192

# Safety gate: no isolated execution backend is available in this notebook.
EXECUTION_BACKEND = "none"  # only "none" is supported
ALLOW_UNSANDBOXED_CODE_EXECUTION = False
TASKS_REQUIRING_EXECUTION = {
    "code_generation",
    "code_refactoring",
    "unit_test_generation",
}
assert EXECUTION_BACKEND == "none"
assert ALLOW_UNSANDBOXED_CODE_EXECUTION is False

print("Model:", MODEL_ID, "| short:", SHORT, "| model_dir:", MODEL_DIR)
print("Model source:", MODEL_SOURCE)
print("MODE:", MODE, "| results subdir:", RESULTS_MODE_DIR)
print("Single GPU:", SINGLE_GPU, "| device_map:", DEVICE_MAP)
print("Execution backend:", EXECUTION_BACKEND, "| generated code executed:", ALLOW_UNSANDBOXED_CODE_EXECUTION)

# Unique run identifier: stable within this kernel session (so re-running
# this cell resumes the same run), unique across sessions/accounts (so a
# later session never overwrites an earlier download). Stored per
# (MODE, MODEL_DIR) so switching models creates a new run directory.
try:
    RUN_IDS
except NameError:
    RUN_IDS = {}
_run_key = (MODE, MODEL_DIR)
if _run_key not in RUN_IDS:
    _stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    RUN_IDS[_run_key] = f"{MODEL_DIR}_{MODE}_{_stamp}_{socket.gethostname()}_{uuid.uuid4().hex[:8]}"
    print("Generated RUN_ID =", RUN_IDS[_run_key])
else:
    print("Reusing RUN_ID =", RUN_IDS[_run_key])
RUN_ID = RUN_IDS[_run_key]
RUN_DIR = os.path.join(RESULTS_ROOT, RESULTS_MODE_DIR, MODEL_DIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
METADATA_DIR = os.path.join(RESULTS_ROOT, "metadata")
os.makedirs(METADATA_DIR, exist_ok=True)
assert RUN_DIR.startswith(RESULTS_ROOT + os.sep), RUN_DIR
print("RUN_ID =", RUN_ID)
print("RUN_DIR =", RUN_DIR)


## 4. Load prompt registry and datasets (read-only)

The prompt registry and both test splits are read from the attached
`/kaggle/input` dataset. Nothing is modified.


In [ ]:
# Load prompt registry
import importlib.util
spec = importlib.util.spec_from_file_location("prompt_registry", os.path.join(PROMPTS_DIR, "prompt_registry.py"))
prompt_registry = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prompt_registry)
build_messages = prompt_registry.build_messages
apply_chat_template = prompt_registry.apply_chat_template
print("Prompt registry loaded from", PROMPTS_DIR)


In [ ]:
# Load test datasets (model-facing only)
import json

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

test_data = {}
for task in TASKS:
    path = os.path.join(DATASETS_TEST, f"{task}.jsonl")
    rows = load_jsonl(path)
    test_data[task] = rows
    print(task, len(rows))

print("Total test examples:", sum(len(v) for v in test_data.values()))


## 5. Model loading - FP16, exactly one GPU

Target: one NVIDIA T4 16 GB. The three models fit independently in FP16:

| Model | FP16 weights | Expected peak (batch 1) | T4 headroom |
|---|---:|---:|---:|
| Qwen2.5-Coder-1.5B-Instruct | 3.1 GB | ~5.0 GB | ~10.8 GB |
| Qwen2.5-Coder-3B-Instruct | 6.2 GB | ~8.5 GB | ~7.3 GB |
| Phi-4-mini-instruct | 7.7 GB | ~11.5 GB | ~4.3 GB |

These are **static estimates** (weights + KV cache + prefill/activation
overhead). They are not a substitute for a real load, and no model is loaded
during preparation. Phi-4-mini is the tightest: its `eager` attention can
materialise a large prefill matrix, so `MAX_INPUT_TOKENS=8192` bounds the
worst case. The actual peak is measured by `torch.cuda.max_memory_allocated()`.
Only one model is loaded per session; run the unload cell before switching.


In [ ]:
# Load model in FP16 on exactly one GPU.
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Enforce the one-model-at-a-time contract.
if "model" in globals() or "tokenizer" in globals():
    raise RuntimeError(
        "A model/tokenizer is already loaded. Run the unload cell before "
        "loading another model."
    )

# Reduce allocator fragmentation before the first CUDA allocation.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

assert torch.cuda.is_available(), (
    "No GPU is available. In Kaggle, set Settings -> Accelerator -> GPU T4 x1."
)
_gpu_count = torch.cuda.device_count()
print("CUDA device count:", _gpu_count)
for _i in range(_gpu_count):
    _p = torch.cuda.get_device_properties(_i)
    print(f"  GPU {_i}: {_p.name} | {_p.total_memory / 1e9:.2f} GB")

if SINGLE_GPU:
    torch.cuda.set_device(0)
    if _gpu_count > 1:
        print("WARNING: multiple GPUs visible; device_map forces GPU 0 only.")

_gpu_name = torch.cuda.get_device_name(0)
_total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print("Active GPU:", _gpu_name, "| VRAM (GB):", round(_total_vram_gb, 2))
if "T4" not in _gpu_name:
    print("WARNING: target GPU is a T4 16 GB; current GPU is", _gpu_name)
assert _total_vram_gb >= 15.0, (
    f"Expected a >=15 GB GPU (T4 16 GB); found {_total_vram_gb:.2f} GB. "
    "Do not continue on a smaller GPU."
)

meta = MODEL_META[MODEL_ID]
load_kwargs = {
    "torch_dtype": torch.float16,
    "device_map": DEVICE_MAP,
    "trust_remote_code": meta["trust_remote_code"],
    "low_cpu_mem_usage": True,
}
if meta["attn"] is not None:
    load_kwargs["attn_implementation"] = meta["attn"]

print("Loading tokenizer from", MODEL_SOURCE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_SOURCE, trust_remote_code=meta["trust_remote_code"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading model...")
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_SOURCE, **load_kwargs)
model.eval()
model.config.use_cache = True
load_time = time.time() - t0
print(f"Model loaded in {load_time:.1f}s")
print("Model dtype:", next(model.parameters()).dtype)
print("Model device:", next(model.parameters()).device)
if torch.cuda.is_available():
    print("Allocated VRAM after load (GB):", round(torch.cuda.memory_allocated() / 1e9, 3))
    torch.cuda.reset_peak_memory_stats()


## 6. Generation helpers (bounded input, batch size 1)

Prompts are tokenized with `truncation=True, max_length=MAX_INPUT_TOKENS`
(8192) as a memory guard. No benchmark prompt reaches this bound; if one
does, the row records `input_truncated=True` and the benchmark must be
reviewed. Generation is single-example and uses the model's native chat
template. No generated code is executed here.


In [ ]:
# Generation helpers
import time, gc, threading

def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def generate_one(record):
    prompt = apply_chat_template(tokenizer, record)
    full_prompt_tokens = len(tokenizer.encode(prompt, add_special_tokens=False))
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[1]
    input_truncated = full_prompt_tokens > MAX_INPUT_TOKENS
    if input_truncated:
        print(f"  WARNING: prompt truncated from {full_prompt_tokens} to {input_len} tokens")

    from transformers import TextIteratorStreamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        **inputs,
        max_new_tokens=GEN_CONFIG["max_new_tokens"],
        temperature=GEN_CONFIG["temperature"],
        do_sample=GEN_CONFIG["do_sample"],
        top_p=GEN_CONFIG["top_p"],
        repetition_penalty=GEN_CONFIG["repetition_penalty"],
        pad_token_id=tokenizer.pad_token_id,
        streamer=streamer,
    )

    gen_error = []

    def _run_generate():
        try:
            with torch.inference_mode():
                model.generate(**gen_kwargs)
        except Exception as exc:
            gen_error.append(exc)
            try:
                streamer.end()
            except Exception:
                pass

    thread = threading.Thread(target=_run_generate)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_start = time.perf_counter()
    thread.start()
    collected = []
    first_token_time = None
    for chunk in streamer:
        if first_token_time is None and chunk.strip():
            first_token_time = time.perf_counter()
        collected.append(chunk)
    thread.join()
    if gen_error:
        raise gen_error[0]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_end = time.perf_counter()

    output_text = "".join(collected)
    output_len = count_tokens(output_text)
    latency = t_end - t_start
    ttft_val = (first_token_time - t_start) if first_token_time is not None else None
    tps = output_len / latency if latency > 0 else 0.0
    return {
        "output": output_text,
        "input_tokens": int(input_len),
        "output_tokens": int(output_len),
        "latency_s": round(latency, 4),
        "ttft_s": round(ttft_val, 4) if ttft_val is not None else None,
        "tokens_per_s": round(tps, 2),
        "input_truncated": bool(input_truncated),
    }


## 7. Evaluation helpers - static checks only

**Safety gate:** `EXECUTION_BACKEND="none"`. `run_python()` refuses to run any
generated program. The three tasks that require executing model output
(`code_generation`, `code_refactoring`, `unit_test_generation`) are still
generated, but their evaluation is recorded as:

```json
{"skipped": true, "reason": "no isolated code-execution backend; generated code was not run"}
```

Only `ast.parse()` (syntax) and text metrics are computed. The other three
tasks (explanation, bug detection, completion) are fully scored.


In [ ]:
# Evaluation helpers - no generated code is executed.
import ast
import os
import re
import difflib
import json as _json

assert EXECUTION_BACKEND == "none", (
    "This notebook only supports EXECUTION_BACKEND='none'. "
    "Do not enable unsandboxed execution."
)

try:
    import rouge_score  # noqa: F401
except ImportError as exc:
    raise RuntimeError(
        "rouge_score is required for code_explanation; install it with "
        "!pip install -q rouge_score before running."
    ) from exc

def extract_code_block(text):
    m = re.search(r"```(?:python|py|python3)?\s*\n?(.*?)```", text, re.DOTALL | re.IGNORECASE)
    if m:
        return m.group(1).strip("\n")
    return text.strip()

def _static_syntax_ok(code):
    # Parse only; never execute the code.
    try:
        ast.parse(code)
        return True
    except (SyntaxError, ValueError):
        return False

def run_python(program, timeout=10):
    # Hard block: no isolated execution backend is available.
    raise RuntimeError(
        "Refusing to execute generated code: no isolated sandbox backend is "
        "configured. This notebook performs static checks only. Do not replace "
        "this guard with an unsandboxed subprocess."
    )

def _execution_skipped():
    return {
        "skipped": True,
        "reason": "no isolated code-execution backend; generated code was not run",
        "execution_backend": EXECUTION_BACKEND,
    }

def load_private(record_id, task):
    for line in open(os.path.join(DATASETS_PRIVATE_TEST, f"{task}.jsonl"), encoding="utf-8"):
        r = _json.loads(line)
        if r["id"] == record_id:
            return r
    raise KeyError(record_id)

def eval_code_generation(record, output):
    code = extract_code_block(output)
    result = _execution_skipped()
    result["compile_ok"] = _static_syntax_ok(code)  # static syntax only
    result["static_only"] = True
    return result

def eval_code_explanation(record, output):
    priv = load_private(record["id"], "code_explanation")
    ref = priv["expected_output"]
    try:
        from rouge_score import rouge_scorer
    except ImportError as exc:
        raise RuntimeError(
            "rouge_score is required for code_explanation; install it with "
            "!pip install -q rouge_score before running."
        ) from exc
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_l = scorer.score(ref, output)["rougeL"].fmeasure
    headings = ["## Purpose", "## Parameters", "## Return value", "## Complexity"]
    heading_ok = all(h in output for h in headings)
    return {"rouge_l": rouge_l, "headings_ok": heading_ok}

def eval_bug_detection(record, output):
    try:
        obj = _json.loads(output.strip())
        pred = bool(obj.get("has_bug"))
        parse_ok = True
    except Exception:
        m = re.search(r"\{.*\}", output, re.DOTALL)
        try:
            obj = _json.loads(m.group(0)) if m else {}
            pred = bool(obj.get("has_bug"))
            parse_ok = True
        except Exception:
            pred = None
            parse_ok = False
    priv = load_private(record["id"], "bug_detection")
    truth = bool(priv["expected_output"])
    tp = fp = tn = fn = 0
    if pred is not None:
        if pred and truth: tp = 1
        elif pred and not truth: fp = 1
        elif (not pred) and (not truth): tn = 1
        else: fn = 1
    return {"parse_ok": parse_ok, "pred": pred, "truth": truth, "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def eval_code_refactoring(record, output):
    code = extract_code_block(output)
    result = _execution_skipped()
    result["compile_ok"] = _static_syntax_ok(code)  # static syntax only
    result["static_only"] = True
    result["changed"] = (re.sub(r"\s+", "", code) != re.sub(r"\s+", "", record["input"]))
    return result

def eval_unit_test_generation(record, output):
    tests = extract_code_block(output)
    result = _execution_skipped()
    result["syntax_ok"] = _static_syntax_ok(tests)
    result["n_tests"] = len(re.findall(r"def\s+test_\w+", tests))
    result["static_only"] = True
    return result

def eval_code_completion(record, output):
    pred = output.strip().splitlines()[0].strip() if output.strip() else ""
    priv = load_private(record["id"], "code_completion")
    target = priv["expected_output"]
    norm_pred = re.sub(r"\s+", "", pred)
    norm_target = re.sub(r"\s+", "", target)
    exact = norm_pred == norm_target
    sim = difflib.SequenceMatcher(None, pred, target).ratio()
    return {"pred": pred, "target": target, "exact_match": exact, "edit_similarity": round(sim, 4)}


## 8. Checkpointing, run metadata and SHA256 helpers

Each example is appended to `RUN_DIR/<task>_results.jsonl` immediately and
flushed. If the notebook is interrupted, re-run this cell and the main loop:
already-written IDs are skipped, so the run resumes within the session.
`RUN_DIR` is unique per session, so results from another account/session
(which live in a different directory or have already been downloaded) are
never overwritten. `SHA256SUMS.txt` is written before export for every
result and metadata file.

The run metadata records `MODE`, `RESULTS_MODE_DIR` (`baseline/` for
`MODE="full"`), the model source, the single-GPU device map, the read-only
input flag and the execution-safety gate.


In [ ]:
# Checkpointing, run metadata and SHA256 helpers
# Every row is appended to RUN_DIR/<task>_results.jsonl immediately so an
# interrupted run resumes instead of restarting. RUN_DIR is unique per
# session, so rows from other sessions can never be overwritten.
import hashlib, json, os as _os

def sha256_file(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def result_path(task):
    return _os.path.join(RUN_DIR, f"{task}_results.jsonl")

def _read_result_rows(task):
    rows = []
    path = result_path(task)
    if not _os.path.exists(path):
        return rows
    with open(path, encoding="utf-8") as fh:
        for line_no, line in enumerate(fh, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                # A truncated last line after an interruption is not counted
                # as done; the example will be retried on resume.
                print(f"  WARN {path}:{line_no}: ignoring unparsable line")
    return rows

def load_done_ids(task):
    return {r["id"] for r in _read_result_rows(task) if "id" in r}

def append_result(task, row):
    path = result_path(task)
    with open(path, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(row, ensure_ascii=False) + "\n")
        fh.flush()
        try:
            _os.fsync(fh.fileno())
        except OSError:
            pass
    write_progress()

def dataset_manifest():
    manifest = {}
    for task in TASKS:
        for base, label in ((DATASETS_TEST, "test"), (DATASETS_PRIVATE_TEST, "private_test")):
            p = _os.path.join(base, f"{task}.jsonl")
            if _os.path.exists(p):
                manifest[f"{label}/{task}"] = {
                    "path": p,
                    "sha256": sha256_file(p),
                    "bytes": _os.path.getsize(p),
                }
    return manifest

def run_metadata():
    import platform as _pl, sys as _sys
    gpu = None
    try:
        import torch as _t
        if _t.cuda.is_available():
            gpu = _t.cuda.get_device_name(0)
    except Exception:
        pass
    return {
        "run_id": RUN_ID,
        "mode": MODE,
        "mode_dir": RESULTS_MODE_DIR,
        "model_id": MODEL_ID,
        "model_source": MODEL_SOURCE,
        "short": SHORT,
        "model_dir": MODEL_DIR,
        "seed": SEED,
        "gen_config": GEN_CONFIG,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "platform": PLATFORM,
        "benchmark_root": BENCHMARK_ROOT,
        "benchmark_root_read_only": INPUT_IS_READ_ONLY,
        "results_root": RESULTS_ROOT,
        "checkpoints_root": CHECKPOINTS_ROOT,
        "zip_dir": ZIP_DIR,
        "run_dir": RUN_DIR,
        "single_gpu": SINGLE_GPU,
        "device_map": str(DEVICE_MAP),
        "execution_backend": EXECUTION_BACKEND,
        "allow_unsandboxed_code_execution": ALLOW_UNSANDBOXED_CODE_EXECUTION,
        "tasks_requiring_execution": sorted(TASKS_REQUIRING_EXECUTION),
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": _sys.version,
        "platform_str": _pl.platform(),
        "gpu": gpu,
        "tasks": TASKS,
        "smoke_per_task": SMOKE_PER_TASK if MODE == "smoke_test" else None,
        "datasets": dataset_manifest(),
        "note": "Kaggle-only temporary filesystem. No generated code is executed. Download the ZIP before disconnecting.",
    }

def write_run_metadata():
    import json as _j
    meta = run_metadata()
    out = _os.path.join(METADATA_DIR, f"run_{RUN_ID}.json")
    with open(out, "w", encoding="utf-8") as fh:
        _j.dump(meta, fh, ensure_ascii=False, indent=2)
        fh.write("\n")
    local = _os.path.join(RUN_DIR, "metadata.json")
    with open(local, "w", encoding="utf-8") as fh:
        _j.dump(meta, fh, ensure_ascii=False, indent=2)
        fh.write("\n")
    # Dataset manifest alongside the run metadata.
    dm_out = _os.path.join(METADATA_DIR, "dataset_manifest.json")
    with open(dm_out, "w", encoding="utf-8") as fh:
        _j.dump(meta["datasets"], fh, ensure_ascii=False, indent=2)
        fh.write("\n")
    return out

def write_progress():
    import json as _j
    counts = {}
    for task in TASKS:
        rows = _read_result_rows(task)
        counts[task] = {
            "rows": len(rows),
            "errors": sum(1 for r in rows if "error" in r),
            "skipped": sum(1 for r in rows if (r.get("eval") or {}).get("skipped")),
        }
    out = _os.path.join(METADATA_DIR, f"progress_{RUN_ID}.json")
    with open(out, "w", encoding="utf-8") as fh:
        _j.dump({
            "run_id": RUN_ID, "mode": MODE, "model": MODEL_ID,
            "counts": counts, "updated_utc": datetime.now(timezone.utc).isoformat(),
        }, fh, ensure_ascii=False, indent=2)
        fh.write("\n")
    return out

def aggregate_csv():
    import pandas as pd
    rows = []
    for task in TASKS:
        for r in _read_result_rows(task):
            ev = r.get("eval", {})
            rows.append({
                "model": SHORT,
                "model_dir": MODEL_DIR,
                "mode": MODE,
                "run_id": RUN_ID,
                "task": task,
                "id": r.get("id"),
                "input_tokens": r.get("input_tokens"),
                "output_tokens": r.get("output_tokens"),
                "latency_s": r.get("latency_s"),
                "ttft_s": r.get("ttft_s"),
                "tokens_per_s": r.get("tokens_per_s"),
                "vram_gb": r.get("vram_gb"),
                "input_truncated": r.get("input_truncated"),
                "eval_skipped": bool(ev.get("skipped")),
                "model_source": r.get("model_source"),
                "eval_json": json.dumps(ev, ensure_ascii=False),
            })
    if rows:
        df = pd.DataFrame(rows)
        out = _os.path.join(RUN_DIR, f"{SHORT}_all_results.csv")
        df.to_csv(out, index=False)
        print("Wrote", out, len(df), "rows")
        return out
    return None

def write_sha256sums(root, filename="SHA256SUMS.txt"):
    entries = {}
    for dirpath, _dirs, fnames in _os.walk(root):
        for fn in sorted(fnames):
            if fn == filename:
                continue
            p = _os.path.join(dirpath, fn)
            rel = _os.path.relpath(p, root).replace(_os.sep, "/")
            entries[rel] = sha256_file(p)
    out = _os.path.join(root, filename)
    with open(out, "w", encoding="utf-8", newline="\n") as fh:
        for rel in sorted(entries):
            fh.write(f"{entries[rel]}  {rel}\n")
    return out, entries

write_run_metadata()
write_progress()
print("RUN_DIR =", RUN_DIR)
print("Metadata in", METADATA_DIR)


## 9. Main benchmark loop (resumable within the session)

Runs every task for the selected model. `MODE="full"` processes all 670 test
examples; `MODE="smoke_test"` processes only the first `SMOKE_PER_TASK` rows
per task (identical order). Smoke results live under `results/smoke_test/`
and official results under `results/baseline/`.

**No generated code is executed.** The three execution-dependent tasks are
still generated, but their `eval` object is `{"skipped": true, ...}`. Re-run
this cell after an interruption: examples already in
`RUN_DIR/<task>_results.jsonl` are skipped. A disconnection destroys the
temporary filesystem - export and download frequently.


In [ ]:
# Main benchmark loop - runs ALL tasks for the selected model
# MODE == "full" runs every test example.
# MODE == "smoke_test" runs only the first SMOKE_PER_TASK rows per task
# (same order as test_data), so smoke results never mix with official ones.
import time

EVAL_FUNCS = {
    "code_generation": eval_code_generation,
    "code_explanation": eval_code_explanation,
    "bug_detection": eval_bug_detection,
    "code_refactoring": eval_code_refactoring,
    "unit_test_generation": eval_unit_test_generation,
    "code_completion": eval_code_completion,
}

print("Execution backend:", EXECUTION_BACKEND)
print("Generated code executed:", ALLOW_UNSANDBOXED_CODE_EXECUTION)
print("Tasks with skipped scoring:", sorted(TASKS_REQUIRING_EXECUTION))

write_run_metadata()
for task in TASKS:
    rows = test_data[task]
    if MODE == "smoke_test":
        rows = rows[:SMOKE_PER_TASK]
    done = load_done_ids(task)
    print(f"\n=== {task}: {len(rows)} total, {len(done)} done, {len(rows)-len(done)} remaining ===")
    if task in TASKS_REQUIRING_EXECUTION:
        print("  NOTE: evaluation will be marked skipped (no isolated backend).")
    for i, record in enumerate(rows, 1):
        if record["id"] in done:
            continue
        try:
            gen = generate_one(record)
            ev = EVAL_FUNCS[task](record, gen["output"])
            vram = None
            if torch.cuda.is_available():
                vram = round(torch.cuda.max_memory_allocated() / 1e9, 3)
            result_row = {
                "id": record["id"],
                "task": task,
                "model": MODEL_ID,
                "model_source": MODEL_SOURCE,
                "model_dir": MODEL_DIR,
                "mode": MODE,
                "run_id": RUN_ID,
                "input_tokens": gen["input_tokens"],
                "output_tokens": gen["output_tokens"],
                "latency_s": gen["latency_s"],
                "ttft_s": gen["ttft_s"],
                "tokens_per_s": gen["tokens_per_s"],
                "input_truncated": gen["input_truncated"],
                "vram_gb": vram,
                "output": gen["output"],
                "eval_skipped": bool(ev.get("skipped")),
                "eval": ev,
            }
            append_result(task, result_row)
            print(f"  [{i}/{len(rows)}] {record['id']} ok | lat={gen['latency_s']}s tps={gen['tokens_per_s']}")
        except Exception as e:
            err_row = {
                "id": record["id"], "task": task, "model": MODEL_ID,
                "model_source": MODEL_SOURCE,
                "model_dir": MODEL_DIR, "mode": MODE, "run_id": RUN_ID,
                "error": str(e),
            }
            append_result(task, err_row)
            print(f"  [{i}/{len(rows)}] {record['id']} FAILED: {e}")
    aggregate_csv()
    write_progress()

aggregate_csv()
write_progress()
write_run_metadata()
print("Done. RUN_DIR =", RUN_DIR)
print("No generated code was executed.")


## 10. Model unload - required before switching models

Run this cell after each model and before loading the next one. Only one model
is kept in GPU memory at a time.


In [ ]:
# Unload model and free GPU memory
import gc
import torch

for _name in ("model", "tokenizer"):
    try:
        del globals()[_name]
    except KeyError:
        pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
print("Model unloaded. GPU memory released.")


## 11. Export ZIP + SHA256 (Kaggle Output pane)

Creates `slm-benchmark_results_<MODE>_<RUN_ID>.zip` containing the whole
`results/` tree plus `SHA256SUMS.txt` and metadata. The ZIP and its `.sha256`
sidecar are written under `/kaggle/working/` and downloaded from the Kaggle
Output pane. Nothing is uploaded to a cloud drive and no browser download is
triggered from the notebook.

The ZIP hash is printed below - copy it next to the downloaded file on
Windows to verify the transfer.


In [ ]:
# Export ZIP for Kaggle Output pane. No cloud drive, no browser download.
import shutil
import os as _os2

assert RESULTS_ROOT.startswith(WORKDIR + _os2.sep), RESULTS_ROOT
assert ZIP_DIR == WORKDIR, ZIP_DIR

aggregate_csv()
write_run_metadata()
write_progress()

# Write the export manifest, then hash everything (manifest included).
zip_name = f"slm-benchmark_results_{MODE}_{RUN_ID}.zip"
_manifest = {
    "run_id": RUN_ID, "mode": MODE, "model_id": MODEL_ID, "short": SHORT,
    "model_dir": MODEL_DIR, "zip_name": zip_name, "platform": PLATFORM,
    "results_root": RESULTS_ROOT, "zip_dir": ZIP_DIR,
    "execution_backend": EXECUTION_BACKEND,
    "generated_code_executed": ALLOW_UNSANDBOXED_CODE_EXECUTION,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
with open(_os2.path.join(METADATA_DIR, "export_manifest.json"), "w", encoding="utf-8") as fh:
    import json as _j2
    _j2.dump(_manifest, fh, ensure_ascii=False, indent=2)
    fh.write("\n")

sums_path, entries = write_sha256sums(RESULTS_ROOT)
print("Wrote", sums_path, f"({len(entries)} files)")
shutil.copyfile(sums_path, _os2.path.join(METADATA_DIR, "SHA256SUMS.txt"))

print("\nResult files under RESULTS_ROOT =", RESULTS_ROOT)
for _dirpath, _dirs, _fnames in _os2.walk(RESULTS_ROOT):
    for _fn in sorted(_fnames):
        _p = _os2.path.join(_dirpath, _fn)
        print(" ", _os2.path.relpath(_p, RESULTS_ROOT), _os2.path.getsize(_p), "bytes")

# Place the ZIP next to RESULTS_ROOT (in /kaggle/working/) so it is not inside
# the tree being archived. It contains a top-level results/ directory.
zip_path = _os2.path.join(ZIP_DIR, zip_name)
_results_parent = _os2.path.dirname(RESULTS_ROOT.rstrip("/")) or "."
_results_base = _os2.path.basename(RESULTS_ROOT.rstrip("/"))
shutil.make_archive(zip_path[:-4], "zip", root_dir=_results_parent, base_dir=_results_base)
print("Zipped:", zip_path, _os2.path.getsize(zip_path), "bytes")

zip_hash = sha256_file(zip_path)
print("ZIP SHA256 =", zip_hash)
sidecar = zip_path + ".sha256"
with open(sidecar, "w", encoding="utf-8", newline="\n") as fh:
    fh.write(zip_hash + "  " + zip_name + "\n")
print("Sidecar:", sidecar)

print("Kaggle: open the Output pane on the right and download:", zip_name)
print("Also download the .sha256 sidecar for verification.")
print("No browser download is triggered from the notebook.")


## 12. Download and merge locally on Windows

The notebook never touches your PC filesystem - you organise the downloads.
`MODE="full"` is stored under `results/baseline/`; `MODE="smoke_test"` under
`results/smoke_test/`. Every session produces a unique `<RUN_ID>`, so merging
is a union: run directories are never overwritten, and duplicate example IDs
from separate sessions are kept in separate run folders.

On Windows (PowerShell):

```powershell
# 1. Create the destination and keep each session ZIP separate
$base = "$env:USERPROFILE\Downloads\slm-benchmark"
New-Item -ItemType Directory -Force -Path $base | Out-Null

# 2. Set the exact ZIP downloaded from the Output pane (name contains RUN_ID)
#    Repeat per session/account with a new session folder and ZIP filename:
$zip = "$base\slm-benchmark_results_full_<RUN_ID>.zip"
Expand-Archive -Path $zip -DestinationPath "$base\session-01" -Force

# 3. Verify the ZIP hash printed in the notebook Output (Get-FileHash)
Get-FileHash $zip -Algorithm SHA256

# 4. Verify per-file hashes and merge (run from the benchmark checkout)
python benchmark\scripts\merge_results.py --source "$base\session-01\results" --dest "$base\merged\results"
# Repeat --source for each extracted session, or pass all sources in one command.
```

If you ran on several Kaggle accounts, extract each account's ZIP to its own
`session-*` folder, then merge all `results/` trees into
`%USERPROFILE%\Downloads\slm-benchmark\merged\results\`. Verify with:

```powershell
python benchmark\scripts\validate_baseline_results.py --results "$base\merged\results\baseline" --check-sha256
python benchmark\scripts\aggregate_baseline.py --results "$base\merged\results\baseline"
```

Full details: `benchmark/docs/RESULTS_DOWNLOAD_AND_MERGE.md`. If a session
disconnects before you download, its temporary files are gone - re-run that
model; the new `RUN_ID` blocks a silent overwrite of the earlier ZIP.
